In [0]:
silver_table = "workspace.default.silver_online_retail"
print("Row count in Silver:", spark.table(silver_table).count())
spark.table(silver_table).printSchema()

In [0]:
from pyspark.sql import functions as F

date_range = spark.sql("""
    SELECT explode(sequence(
        (SELECT min(to_date(InvoiceDate)) FROM workspace.default.silver_online_retail),
        (SELECT max(to_date(InvoiceDate)) FROM workspace.default.silver_online_retail),
        interval 1 day
    )) AS full_date
""")

dim_date = date_range.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    "full_date",
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.weekofyear("full_date").alias("week_of_year"),
    F.dayofmonth("full_date").alias("day_of_month"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").alias("day_of_week"),
    (F.dayofweek("full_date").isin([1, 7])).alias("is_weekend")
)

display(dim_date.limit(10))
print("dim_date row count:", dim_date.count())

In [0]:
dim_date.write.mode("overwrite").saveAsTable("workspace.default.dim_date")

print("dim_date written. Row count:", spark.table("workspace.default.dim_date").count())

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

customer_country_counts = spark.sql("""
    SELECT
        Customer_ID AS customer_id,
        Country AS country,
        COUNT(*) AS cnt
    FROM workspace.default.silver_online_retail
    WHERE is_null_customer = false
    GROUP BY Customer_ID, Country
""")

window_spec = Window.partitionBy("customer_id").orderBy(F.desc("cnt"))

dim_customer = (
    customer_country_counts
    .withColumn("rn", F.row_number().over(window_spec))
    .filter("rn = 1")
    .select("customer_id", "country")
)

display(dim_customer.limit(10))
print("dim_customer row count:", dim_customer.count())

dim_customer.groupBy("customer_id").agg(F.count("*").alias("cnt")).filter("cnt > 1").show()

In [0]:
dim_customer.write.mode("overwrite").saveAsTable("workspace.default.dim_customer")

print("dim_customer written. Row count:", spark.table("workspace.default.dim_customer").count())

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

product_desc_counts = spark.sql("""
    SELECT
        StockCode AS stock_code,
        Description AS description,
        COUNT(*) AS cnt
    FROM workspace.default.silver_online_retail
    WHERE is_invalid_price = false
      AND Description IS NOT NULL
    GROUP BY StockCode, Description
""")

window_spec = Window.partitionBy("stock_code").orderBy(F.desc("cnt"))

dim_product = (
    product_desc_counts
    .withColumn("rn", F.row_number().over(window_spec))
    .filter("rn = 1")
    .select("stock_code", "description")
)

display(dim_product.limit(10))
print("dim_product row count:", dim_product.count())

dim_product.groupBy("stock_code").agg(F.count("*").alias("cnt")).filter("cnt > 1").show()

In [0]:
dim_product.write.mode("overwrite").saveAsTable("workspace.default.dim_product")

print("dim_product written. Row count:", spark.table("workspace.default.dim_product").count())

In [0]:
from pyspark.sql import functions as F

fact_sales = spark.sql("""
    SELECT
        Invoice AS invoice_no,
        StockCode AS stock_code,
        Customer_ID AS customer_id,
        date_format(to_date(InvoiceDate), 'yyyyMMdd') AS date_key,
        Quantity AS quantity,
        Price AS unit_price,
        line_total,
        is_cancelled,
        is_return
    FROM workspace.default.silver_online_retail
    WHERE is_null_customer = false
      AND is_invalid_price = false
      AND is_duplicate = false
""")

fact_sales = fact_sales.withColumn("date_key", fact_sales["date_key"].cast("int"))

display(fact_sales.limit(10))
print("fact_sales row count:", fact_sales.count())
print("  of which cancelled/returns:", fact_sales.filter("is_cancelled = true OR is_return = true").count())

In [0]:
from pyspark.sql import functions as F

# orphan check: fact rows whose customer_id doesn't exist in dim_customer
orphan_customers = fact_sales.join(
    spark.table("workspace.default.dim_customer"),
    on="customer_id",
    how="left_anti"
)
print("Orphan customer_id rows:", orphan_customers.count())

# orphan check: fact rows whose stock_code doesn't exist in dim_product
orphan_products = fact_sales.join(
    spark.table("workspace.default.dim_product"),
    on="stock_code",
    how="left_anti"
)
print("Orphan stock_code rows:", orphan_products.count())

# orphan check: fact rows whose date_key doesn't exist in dim_date
orphan_dates = fact_sales.join(
    spark.table("workspace.default.dim_date"),
    on="date_key",
    how="left_anti"
)
print("Orphan date_key rows:", orphan_dates.count())

In [0]:
fact_sales.write.mode("overwrite").saveAsTable("workspace.default.fact_sales")

print("fact_sales written. Row count:", spark.table("workspace.default.fact_sales").count())

In [0]:
print("dim_date:", spark.table("workspace.default.dim_date").count())
print("dim_customer:", spark.table("workspace.default.dim_customer").count())
print("dim_product:", spark.table("workspace.default.dim_product").count())
print("fact_sales:", spark.table("workspace.default.fact_sales").count())

# quick sanity check: total net line_total should be a reasonable-looking revenue figure
from pyspark.sql import functions as F
totals = spark.table("workspace.default.fact_sales").agg(
    F.sum("line_total").alias("gross_total"),
    F.sum(F.when(F.col("is_cancelled") == False, F.col("line_total")).otherwise(0)).alias("net_total")
)
totals.show()